# Messi Goals — Visualizations (my 3 of the team's 9)
**Platform:** Python (pandas + Plotly). **LLM:** Claude was used for cleaning code and chart code; see the *Prompt log* at the end.

Data: `messi_goals.csv` (one row per goal, 2004/05–2022/23). Place the file next to this notebook, or change `DATA_PATH`.

## 0. Load and clean
Fixes Excel damage (seasons read as dates, scores read as times), flips away-game scores, and builds numeric helper columns.

In [1]:
import re
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

DATA_PATH = "data/messi_goals.csv"

df = pd.read_csv(DATA_PATH, encoding_errors="replace")
df.columns = df.columns.str.strip()
for c in df.columns:
    df[c] = df[c].astype("string").str.strip()

# Excel turned "11/12" and "12/13" into dates
df["Season"] = df["Season"].replace({"11-Dec": "11/12", "Dec-13": "12/13"})
df["Competition"] = df["Competition"].replace({"Champions League": "UEFA Champions League"})
df["Competition"] = df["Competition"].str.replace(r"Troph.{1,3}e des Champions", "Trophée des Champions", regex=True)

# "2:00" really means 2-0 (Excel read it as h:mm); "5:4 AET" has a suffix
def score(s):
    a, b = re.match(r"(\d+):(\d+)", s).groups()
    return int(a), int(b)
df[["team_goals", "opp_goals"]] = df["Result"].apply(lambda s: pd.Series(score(s)))
away = df["Venue"] == "A"                       # Result is written home:away
df.loc[away, ["team_goals", "opp_goals"]] = df.loc[away, ["opp_goals", "team_goals"]].values

def minute(m):                                  # "90+1" -> 91
    p = str(m).split("+")
    return int(p[0]) + (int(p[1]) if len(p) > 1 else 0)
df["minute_num"] = df["Minute"].apply(minute)
df["season_start"] = 2000 + df["Season"].str[:2].astype(int)
df["Type"] = df["Type"].fillna("Unknown")
df["Goal_assist"] = df["Goal_assist"].fillna("Unassisted")
print(df.shape)
df.head()

(704, 17)


,Season,Competition,Matchday,Date,Venue,Club,Opponent,Result,Playing_Position,Minute,At_score,Type,Goal_assist,team_goals,opp_goals,minute_num,season_start
0,04/05,LaLiga,34,05-01/05,H,FC Barcelona,Albacete Balompie,2:00,CF,90+1,2:00,Left-footed shot,Ronaldinho Gaacho,2,0,91,2004
1,05/06,UEFA Champions League,Group Stage,11-02/05,H,FC Barcelona,Panathinaikos Athens,5:00,RW,34,3:00,Left-footed shot,Unassisted,5,0,34,2005
2,05/06,LaLiga,13,11/27/05,H,FC Barcelona,Racing Santander,4:01,RW,51,2:00,Left-footed shot,Samuel Etoo,4,1,51,2005
3,05/06,LaLiga,19,1/15/06,H,FC Barcelona,Athletic Bilbao,2:01,RW,50,2:01,Left-footed shot,Mark van Bommel,2,1,50,2005
4,05/06,LaLiga,20,1/22/06,H,FC Barcelona,Deportivo Alaves,2:00,CF,82,2:00,Left-footed shot,Ronaldinho Gaacho,2,0,82,2005


## V1 — Goals per season, stacked by competition
**Data feature:** time series with a categorical breakdown. **Question:** how did his output peak and where did the goals come from?
*Chart type:* stacked bar (Plotly Express).

In [2]:
v1 = (df.groupby(["Season", "season_start", "Competition"]).size()
        .reset_index(name="goals").sort_values("season_start"))
fig1 = px.bar(v1, x="Season", y="goals", color="Competition",
              title="Messi goals per season by competition",
              labels={"goals": "Goals"},
              category_orders={"Season": v1["Season"].drop_duplicates().tolist()})
fig1.update_layout(barmode="stack", legend_title_text="Competition", hovermode="x unified")
fig1.show()

## V2 — When in the match he scores, by season (heatmap)
**Data feature:** continuous minute × season. **Question:** does he score late more in some eras?
*Chart type:* heatmap built with `go.Heatmap` from 10-minute bins (I do the binning myself).

In [3]:
bins = list(range(0, 100, 10)) + [130]
labels = [f"{b}-{b+9}" for b in range(0, 90, 10)] + ["90+"]
df["minute_bin"] = pd.cut(df["minute_num"], bins=bins, labels=labels, right=False)

pivot = (df.pivot_table(index="Season", columns="minute_bin", values="Opponent",
                        aggfunc="count", observed=False, fill_value=0)
           .sort_index())

fig2 = go.Figure(go.Heatmap(
    z=pivot.values, x=pivot.columns.astype(str), y=pivot.index,
    colorscale="YlOrRd", colorbar=dict(title="Goals"),
    hovertemplate="Season %{y}<br>Minute %{x}<br>Goals: %{z}<extra></extra>"))
fig2.update_layout(title="When Messi scores: minute window vs season",
                   xaxis_title="Minute of match", yaxis_title="Season",
                   yaxis=dict(autorange="reversed"))
fig2.show()

## V3: Opponents and match environment (home vs. away)
**Data feature:** categorical comparison with a two-level split. **Question:** which opponents did he score against most, and did the venue matter?
*Chart type:* stacked horizontal bar of the 10 most-scored-against opponents, split by home and away (Plotly Express).

In [4]:
top_opp = df["Opponent"].value_counts().head(10).index
v3 = (df[df["Opponent"].isin(top_opp)]
        .groupby(["Opponent", "Venue"]).size().reset_index(name="goals"))
v3["Venue"] = v3["Venue"].map({"H": "Home", "A": "Away"})
order = df["Opponent"].value_counts().head(10).index.tolist()[::-1]   # largest at top

fig3 = px.bar(v3, x="goals", y="Opponent", color="Venue", orientation="h",
              title="Messi goals against his 10 most frequent victims, home vs away",
              labels={"goals": "Goals"},
              category_orders={"Opponent": order, "Venue": ["Home", "Away"]})
fig3.update_layout(barmode="stack", hovermode="y unified")
fig3.show()

## Case Study 2: The Evolution of Messi's Career

**Research question:** How did Messi's scoring change across seasons, competitions, opponents and match environments?

**User and goal:** A football analyst wants to understand how Messi's scoring output, timing and opposition changed over his 19 seasons, to judge when he was at his peak and how consistent he was.

1. **Step 1 (V1):** Open goals per season. He scores 1 goal in 04/05 and 8 in 05/06, builds to 53 in 10/11, and peaks at 73 in 11/12. After the peak he stays between roughly 40 and 60 goals a season until 18/19, then drops to 31, 38, 14 and 18 in the last four seasons (the final two at PSG, 32 goals).
2. **Step 2 (V1):** Break the peak season down by competition. In 11/12, 50 of the 73 goals came in LaLiga and 14 in the Champions League. Across the whole career LaLiga accounts for 474 of 704 goals and the Champions League for 129.
3. **Step 3 (V2):** Open the minute-by-season heatmap to see how timing changed. About 34% of his goals from 04/05 to 10/11 came from the 75th minute on, falling to 21% in 11/12 to 16/17 and rising again to 26% in 17/18 to 22/23. *Say which seasons show the strongest late-goal columns in the heatmap.*
4. **Step 4 (V3):** Move from time to opposition and environment. Sevilla (38), Atlético de Madrid (32) and Valencia (31) are the opponents he scored against most. For most of them he scores more at home, but Real Madrid is the exception, with 15 away goals against 11 at home. *Say whether this surprised you.*

**Conclusion:** *One or two sentences with a decision, for example whether his peak was a short burst or a long plateau, and what the shift in late goals suggests about his role.*

**Link to featured papers:** *One sentence tying this scenario to a usage scenario from the course papers.*

## Prompt log (required for reproducibility)

| # | Tool/model | Exact prompt | What I kept / changed |
|---|---|---|---|
| 1 | Claude | *Pasted the assignment description plus the course slides and asked for help* | Used the rubric breakdown to plan the notebook |
| 2 | Claude | *Uploaded `messi_goals.csv`* | Used the cleaning script; the raw data had Excel-mangled seasons and scores |
| 3 | Claude | "yes please write the plots for the dataset" | Kept V1-V3 as generated; *note any edits you made* |
| 4 | Claude | *Pasted the `ModuleNotFoundError` and asked how to fix it* | Installed the libraries and set the data path |
| 5 | Claude | *Uploaded the teammate's notebook and asked what else was missing and whether my charts should be reworked* | Replaced my assister Sankey (V3) with an opponents home/away chart because it overlapped the teammate's charts; rewrote my case study as Case Study 2 |